# Global Pollution Analysis and Energy Recovery

### Apriori Association Rule Mining and Validation

**Objective:**  
This project studies the relationship between pollution indicators, energy consumption, renewable energy, and energy recovery across countries and years.

The main technique used is the **Apriori algorithm**. Since Apriori works with transactions and categorical items, the numerical pollution and energy variables are converted into meaningful categories before association-rule mining.

> **Note:** The dataset already contains `Energy_Consumption_Per_Capita (in MWh)`, so it is analyzed directly rather than calculating the same feature again.


### Package check

If `mlxtend` is not installed in your environment, run the following cell once.


In [ ]:
# Uncomment this line only if mlxtend is not already installed.
# %pip install mlxtend


## 1. Importing the required libraries

The libraries below are enough for the complete analysis. `mlxtend` is used for the Apriori algorithm and association rules.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")


## 2. Loading the dataset

In [ ]:
from pathlib import Path

# Keep the CSV in the same folder as this notebook.
file_path = Path("Global_Pollution_Analysis(5).csv")

if not file_path.exists():
    csv_files = list(Path(".").glob("*.csv"))
    if len(csv_files) == 1:
        file_path = csv_files[0]
    else:
        raise FileNotFoundError(
            "Place Global_Pollution_Analysis(5).csv in the notebook folder."
        )

data = pd.read_csv(file_path)

print("Dataset shape:", data.shape)
data.head()


## 3. Basic information about the data

In [ ]:
print("Number of rows:", len(data))
print("Number of columns:", len(data.columns))
print("\nColumn names:")
print(data.columns.tolist())

print("\nData types:")
print(data.dtypes)


## 4. Checking missing values and duplicate records

In [ ]:
missing = data.isnull().sum().sort_values(ascending=False)
print("Missing values:")
print(missing)

print("\nTotal missing values:", data.isnull().sum().sum())
print("Duplicate rows:", data.duplicated().sum())


There are no missing values in the supplied dataset, so no artificial imputation is needed. Duplicate records are also checked before continuing with the analysis.


In [ ]:
# Remove duplicate rows if any are found.
data = data.drop_duplicates().reset_index(drop=True)

print("Shape after duplicate check:", data.shape)


## 5. Descriptive statistics

In [ ]:
numeric_data = data.select_dtypes(include=np.number)
numeric_data.describe().T


## 6. Checking categorical information

In [ ]:
print("Number of countries:", data["Country"].nunique())
print("Year range:", int(data["Year"].min()), "to", int(data["Year"].max()))

print("\nSample countries:")
print(data["Country"].drop_duplicates().head(15).to_list())


## 7. Pollution and energy distributions

In [ ]:
pollution_cols = [
    "Air_Pollution_Index",
    "Water_Pollution_Index",
    "Soil_Pollution_Index"
]

energy_cols = [
    "Energy_Recovered (in GWh)",
    "Energy_Consumption_Per_Capita (in MWh)",
    "Renewable_Energy (%)"
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, pollution_cols):
    ax.hist(data[col], bins=20, edgecolor="black")
    ax.set_title(col)
    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, energy_cols):
    ax.hist(data[col], bins=20, edgecolor="black")
    ax.set_title(col)
    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()


## 8. Correlation analysis

In [ ]:
corr = numeric_data.corr()

plt.figure(figsize=(12, 9))
plt.imshow(corr, cmap="coolwarm", aspect="auto")
plt.colorbar(label="Correlation")

plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)

plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


Correlation is useful here as an exploratory step. It should not be treated as proof of causation. The Apriori section below looks at co-occurring categories instead.


## 9. Pollution trends over time

In [ ]:
yearly = data.groupby("Year")[pollution_cols + ["Energy_Recovered (in GWh)"]].mean()

plt.figure(figsize=(12, 6))

for col in pollution_cols:
    plt.plot(yearly.index, yearly[col], marker="o", label=col)

plt.title("Average Pollution Indices by Year")
plt.xlabel("Year")
plt.ylabel("Average Index")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(yearly.index, yearly["Energy_Recovered (in GWh)"], marker="o")
plt.title("Average Energy Recovered by Year")
plt.xlabel("Year")
plt.ylabel("Energy_Recovered (in GWh)")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 9.1 Normalizing pollution indices

Apriori itself does not require normalization because the final transaction table is categorical. However, normalized versions are useful for exploratory analysis and satisfy the preprocessing requirement.

Min-max normalization puts each pollution index between 0 and 1.


In [ ]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

scaler = MinMaxScaler()

pollution_normalized = scaler.fit_transform(
    data[pollution_cols]
)

normalized_names = [
    "Air Pollution Normalized",
    "Water Pollution Normalized",
    "Soil Pollution Normalized"
]

normalized_df = pd.DataFrame(
    pollution_normalized,
    columns=normalized_names,
    index=data.index
)

data[normalized_names] = normalized_df

data[normalized_names].head()


## 9.2 Encoding categorical features

Country is categorical, so a label-encoded version is created for numerical processing. Year is already numeric and is therefore kept as an integer rather than needlessly encoding it.


In [ ]:
country_encoder = LabelEncoder()
year_encoder = LabelEncoder()

data["Country_Encoded"] = country_encoder.fit_transform(data["Country"])
data["Year_Encoded"] = year_encoder.fit_transform(data["Year"].astype(str))

data[["Country", "Country_Encoded", "Year", "Year_Encoded"]].head()


## 10. Feature engineering

Apriori needs a transaction-like table containing items such as:

- `Air Pollution=High`
- `Water Pollution=Medium`
- `Renewable Energy=High`
- `Energy Recovery=Low`

The thresholds are created from the dataset distribution rather than arbitrary universal pollution limits. This makes the categories suitable for this particular dataset.


In [ ]:
def make_level(series):
    # Using the 33rd and 67th percentiles gives three roughly balanced groups.
    low_cut = series.quantile(0.33)
    high_cut = series.quantile(0.67)

    return pd.cut(
        series,
        bins=[-np.inf, low_cut, high_cut, np.inf],
        labels=["Low", "Medium", "High"]
    )

data["Air Pollution Level"] = make_level(data["Air_Pollution_Index"])
data["Water Pollution Level"] = make_level(data["Water_Pollution_Index"])
data["Soil Pollution Level"] = make_level(data["Soil_Pollution_Index"])

data["Energy Recovery Level"] = make_level(data["Energy_Recovered (in GWh)"])
data["Energy Consumption Level"] = make_level(data["Energy_Consumption_Per_Capita (in MWh)"])
data["Renewable Energy Level"] = make_level(data["Renewable_Energy (%)"])

data[[
    "Air Pollution Level",
    "Water Pollution Level",
    "Soil Pollution Level",
    "Energy Recovery Level",
    "Energy Consumption Level",
    "Renewable Energy Level"
]].head()


### Thresholds used for the categories

In [ ]:
thresholds = pd.DataFrame({
    "Feature": [
        "Air_Pollution_Index",
        "Water_Pollution_Index",
        "Soil_Pollution_Index",
        "Energy_Recovered (in GWh)",
        "Energy_Consumption_Per_Capita (in MWh)",
        "Renewable_Energy (%)"
    ],
    "Low / Medium boundary": [
        data[c].quantile(0.33) for c in [
            "Air_Pollution_Index",
            "Water_Pollution_Index",
            "Soil_Pollution_Index",
            "Energy_Recovered (in GWh)",
            "Energy_Consumption_Per_Capita (in MWh)",
            "Renewable_Energy (%)"
        ]
    ],
    "Medium / High boundary": [
        data[c].quantile(0.67) for c in [
            "Air_Pollution_Index",
            "Water_Pollution_Index",
            "Soil_Pollution_Index",
            "Energy_Recovered (in GWh)",
            "Energy_Consumption_Per_Capita (in MWh)",
            "Renewable_Energy (%)"
        ]
    ]
})

thresholds


## 11. Adding pollution trend information

For each country, the pollution values are compared with the previous available year. This gives a simple trend category:

- Increasing
- Decreasing
- Stable

The first observation for a country does not have a previous year, so its trend is marked as `Unknown`.


In [ ]:
data = data.sort_values(["Country", "Year"]).reset_index(drop=True)

data["Air Pollution Change"] = data.groupby("Country")["Air_Pollution_Index"].diff()

data["Air Pollution Trend"] = np.select(
    [
        data["Air Pollution Change"] > 0,
        data["Air Pollution Change"] < 0
    ],
    [
        "Increasing",
        "Decreasing"
    ],
    default="Stable"
)

data.loc[data["Air Pollution Change"].isna(), "Air Pollution Trend"] = "Unknown"

data[[
    "Country", "Year",
    "Air_Pollution_Index",
    "Air Pollution Change",
    "Air Pollution Trend"
]].head(12)


## 12. Preparing transactions for Apriori

In [ ]:
# Keep the items directly related to the project objective.
transaction_data = pd.DataFrame({
    "Country": data["Country"],
    "Year": data["Year"],
    "Air Pollution": "Air Pollution=" + data["Air Pollution Level"].astype(str),
    "Water Pollution": "Water Pollution=" + data["Water Pollution Level"].astype(str),
    "Soil Pollution": "Soil Pollution=" + data["Soil Pollution Level"].astype(str),
    "Energy Recovery": "Energy Recovery=" + data["Energy Recovery Level"].astype(str),
    "Energy Consumption": "Energy Consumption=" + data["Energy Consumption Level"].astype(str),
    "Renewable Energy": "Renewable Energy=" + data["Renewable Energy Level"].astype(str),
    "Air Trend": "Air Trend=" + data["Air Pollution Trend"].astype(str)
})

transaction_data.head()


The country and year identify the observation but are not included as Apriori items. Including every country name would make the rules mostly describe country frequency instead of pollution-energy relationships.


In [ ]:
item_columns = [
    "Air Pollution",
    "Water Pollution",
    "Soil Pollution",
    "Energy Recovery",
    "Energy Consumption",
    "Renewable Energy",
    "Air Trend"
]

basket = pd.get_dummies(transaction_data[item_columns], dtype=bool)

basket.head()


## 13. Finding frequent itemsets with Apriori

In [ ]:
# Start with a moderate support value.
min_support = 0.10

frequent_itemsets = apriori(
    basket,
    min_support=min_support,
    use_colnames=True
)

frequent_itemsets["item_count"] = frequent_itemsets["itemsets"].apply(len)
frequent_itemsets = frequent_itemsets.sort_values(
    ["support", "item_count"],
    ascending=[False, True]
).reset_index(drop=True)

print("Number of frequent itemsets:", len(frequent_itemsets))
frequent_itemsets.head(20)


### Frequent itemset visualization

In [ ]:
top_itemsets = frequent_itemsets.head(15).copy()
top_itemsets["items"] = top_itemsets["itemsets"].apply(
    lambda x: " + ".join(sorted(x))
)

plt.figure(figsize=(10, 7))
plt.barh(top_itemsets["items"][::-1], top_itemsets["support"][::-1])
plt.xlabel("Support")
plt.title("Top Frequent Itemsets")
plt.tight_layout()
plt.show()


## 14. Generating association rules

In [ ]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.50
)

rules = rules.sort_values(
    ["lift", "confidence", "support"],
    ascending=False
).reset_index(drop=True)

print("Number of rules:", len(rules))

rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]].head(20)


### Understanding the main measures

**Support** tells us how often the complete combination occurs in the dataset.

**Confidence** tells us how often the consequent occurs when the antecedent occurs.

**Lift** compares the observed co-occurrence with what would be expected if the two sides were independent.

- Lift > 1: positive association
- Lift ≈ 1: little evidence of association
- Lift < 1: negative association

A high confidence rule is not automatically useful if the consequent is already very common, which is why lift is considered alongside confidence.


## 15. Filtering the more useful rules

In [ ]:
useful_rules = rules[
    (rules["support"] >= 0.10) &
    (rules["confidence"] >= 0.60) &
    (rules["lift"] > 1.10)
].copy()

print("Rules after filtering:", len(useful_rules))

useful_rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]].head(20)


In [ ]:
def format_rule(row):
    left = ", ".join(sorted(row["antecedents"]))
    right = ", ".join(sorted(row["consequents"]))
    return f"{left}  ->  {right}"

rule_plot = useful_rules.head(15).copy()
rule_plot["rule"] = rule_plot.apply(format_rule, axis=1)

plt.figure(figsize=(11, 7))
plt.barh(rule_plot["rule"][::-1], rule_plot["lift"][::-1])
plt.xlabel("Lift")
plt.title("Association Rules Ranked by Lift")
plt.tight_layout()
plt.show()


## 16. Pollution and energy recovery rules

In [ ]:
pollution_terms = [
    "Air Pollution=",
    "Water Pollution=",
    "Soil Pollution=",
    "Energy Recovery="
]

def has_pollution_energy_item(itemset):
    text = " | ".join(itemset)
    return (
        ("Air Pollution=" in text or
         "Water Pollution=" in text or
         "Soil Pollution=" in text)
        and "Energy Recovery=" in text
    )

focused_rules = rules[
    rules.apply(
        lambda row: has_pollution_energy_item(
            set(row["antecedents"]) | set(row["consequents"])
        ),
        axis=1
    )
].copy()

focused_rules = focused_rules[
    (focused_rules["confidence"] >= 0.50) &
    (focused_rules["lift"] > 1)
].sort_values(
    ["lift", "confidence"],
    ascending=False
)

focused_rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]].head(20)


## 17. Validation using different support thresholds

Apriori does not produce a conventional prediction accuracy. Instead, we can test whether the discovered associations remain reasonably stable when the minimum support is changed.

This helps avoid building conclusions around one arbitrary threshold.


In [ ]:
support_values = [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]

validation_rows = []

for support_value in support_values:
    itemsets = apriori(
        basket,
        min_support=support_value,
        use_colnames=True
    )

    if len(itemsets) == 0:
        validation_rows.append({
            "min_support": support_value,
            "frequent_itemsets": 0,
            "rules": 0
        })
        continue

    temp_rules = association_rules(
        itemsets,
        metric="confidence",
        min_threshold=0.50
    )

    validation_rows.append({
        "min_support": support_value,
        "frequent_itemsets": len(itemsets),
        "rules": len(temp_rules)
    })

validation_table = pd.DataFrame(validation_rows)
validation_table


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    validation_table["min_support"],
    validation_table["frequent_itemsets"],
    marker="o",
    label="Frequent itemsets"
)
plt.plot(
    validation_table["min_support"],
    validation_table["rules"],
    marker="o",
    label="Rules"
)

plt.xlabel("Minimum support")
plt.ylabel("Count")
plt.title("Apriori Sensitivity to Minimum Support")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 18. Train/test-style validation by year

For a time-based dataset, a chronological split is more meaningful than randomly mixing years.

Here, earlier years are used to discover rules and later years are used to check whether the same item combinations continue to occur.

This is not classification accuracy. It is a simple **rule stability check**.


In [ ]:
split_year = int(data["Year"].quantile(0.70))

train_data = data[data["Year"] <= split_year].copy()
test_data = data[data["Year"] > split_year].copy()

print("Training years:", train_data["Year"].min(), "to", train_data["Year"].max())
print("Testing years:", test_data["Year"].min(), "to", test_data["Year"].max())
print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))


In [ ]:
def make_basket(frame):
    temp = pd.DataFrame({
        "Air Pollution": "Air Pollution=" + frame["Air Pollution Level"].astype(str),
        "Water Pollution": "Water Pollution=" + frame["Water Pollution Level"].astype(str),
        "Soil Pollution": "Soil Pollution=" + frame["Soil Pollution Level"].astype(str),
        "Energy Recovery": "Energy Recovery=" + frame["Energy Recovery Level"].astype(str),
        "Energy Consumption": "Energy Consumption=" + frame["Energy Consumption Level"].astype(str),
        "Renewable Energy": "Renewable Energy=" + frame["Renewable Energy Level"].astype(str),
        "Air Trend": "Air Trend=" + frame["Air Pollution Trend"].astype(str)
    })
    return pd.get_dummies(temp, dtype=bool)

train_basket = make_basket(train_data)
test_basket = make_basket(test_data)

# Make sure both baskets have exactly the same item columns.
train_basket, test_basket = train_basket.align(
    test_basket,
    join="outer",
    axis=1,
    fill_value=False
)

train_itemsets = apriori(
    train_basket,
    min_support=0.10,
    use_colnames=True
)

train_rules = association_rules(
    train_itemsets,
    metric="confidence",
    min_threshold=0.50
)

print("Rules discovered in training data:", len(train_rules))


In [ ]:
# Measure how often each training rule's complete item combination occurs in the later period.
def transaction_support(frame_basket, items):
    if not items:
        return 0

    missing = [item for item in items if item not in frame_basket.columns]
    if missing:
        return 0

    return frame_basket[list(items)].all(axis=1).mean()

if len(train_rules) > 0:
    stability = []

    for _, row in train_rules.iterrows():
        test_support = transaction_support(
            test_basket,
            set(row["antecedents"]) | set(row["consequents"])
        )

        stability.append(test_support)

    train_rules["test_combination_support"] = stability

    stable_rules = train_rules[
        train_rules["test_combination_support"] > 0
    ].sort_values(
        "test_combination_support",
        ascending=False
    )

    stable_rules[[
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift",
        "test_combination_support"
    ]].head(20)
else:
    print("No rules were generated with the selected training threshold.")


## 19. Country-level pollution and energy overview

The following table gives a compact country-level summary. It can be used to support the interpretation of the association rules without claiming that the rules prove causation.


In [ ]:
country_summary = data.groupby("Country").agg({
    "Air_Pollution_Index": "mean",
    "Water_Pollution_Index": "mean",
    "Soil_Pollution_Index": "mean",
    "Energy_Recovered (in GWh)": "mean",
    "Energy_Consumption_Per_Capita (in MWh)": "mean",
    "Renewable_Energy (%)": "mean"
}).sort_values("Energy_Recovered (in GWh)", ascending=False)

country_summary.head(15)


## 20. Automatically generating a few observations

In [ ]:
print("Dataset observations")
print("-" * 50)

print(f"Average air pollution index: {data['Air Pollution Index'].mean():.2f}")
print(f"Average water pollution index: {data['Water Pollution Index'].mean():.2f}")
print(f"Average soil pollution index: {data['Soil Pollution Index'].mean():.2f}")
print(f"Average energy recovered: {data['Energy Recovered'].mean():.2f}")
print(f"Average renewable energy share: {data['Renewable Energy %'].mean():.2f}%")

if len(focused_rules) > 0:
    best_rule = focused_rules.iloc[0]

    print("\nStrongest pollution-energy rule by lift:")
    print(format_rule(best_rule))
    print(f"Support: {best_rule['support']:.3f}")
    print(f"Confidence: {best_rule['confidence']:.3f}")
    print(f"Lift: {best_rule['lift']:.3f}")
else:
    print("\nNo pollution-energy rule passed the selected filtering conditions.")


## 20.1 CNN component required by the original assignment

The original assignment mentions a CNN for delivery prediction. The supplied `Global_Pollution_Analysis.csv` contains tabular pollution and energy data and has no image files or delivery target.

A CNN therefore cannot be honestly trained from this CSV. The following cell documents the exact situation instead of generating artificial image data and presenting it as a project result.

If your teacher has provided a separate image/delivery dataset, that dataset should be connected here and the CNN should be trained separately.


In [ ]:
# CNN readiness check
import os

image_dataset_folder = "delivery_images"  # Change this if a separate image dataset is provided.

if os.path.isdir(image_dataset_folder):
    print("A separate image dataset folder was found.")
    print("Use the CNN section below after confirming the folder contains labeled classes.")
else:
    print("No separate image/delivery dataset was supplied with this project.")
    print("CNN training is therefore not run on the pollution CSV.")


### CNN template for a separate image dataset

The following is a ready-to-adapt model template. It is intentionally not executed with the pollution CSV because doing so would mix two unrelated data types.

Expected folder format:

```text
delivery_images/
    class_1/
        image1.jpg
        image2.jpg
    class_2/
        image3.jpg
        image4.jpg
```

The target classes should represent the delivery outcome defined by the separate assignment dataset.


In [ ]:
# Optional CNN template — use only with a real labeled image dataset.

# import tensorflow as tf
# from tensorflow.keras import layers, models
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
# import seaborn as sns

# IMG_SIZE = (128, 128)
# BATCH_SIZE = 32

# train_ds = tf.keras.utils.image_dataset_from_directory(
#     image_dataset_folder,
#     validation_split=0.20,
#     subset="training",
#     seed=42,
#     image_size=IMG_SIZE,
#     batch_size=BATCH_SIZE
# )

# val_ds = tf.keras.utils.image_dataset_from_directory(
#     image_dataset_folder,
#     validation_split=0.20,
#     subset="validation",
#     seed=42,
#     image_size=IMG_SIZE,
#     batch_size=BATCH_SIZE
# )

# class_names = train_ds.class_names
# print("Classes:", class_names)

# cnn_model = models.Sequential([
#     layers.Rescaling(1./255, input_shape=IMG_SIZE + (3,)),
#     layers.Conv2D(32, 3, activation="relu"),
#     layers.MaxPooling2D(),
#     layers.Conv2D(64, 3, activation="relu"),
#     layers.MaxPooling2D(),
#     layers.Conv2D(128, 3, activation="relu"),
#     layers.MaxPooling2D(),
#     layers.Flatten(),
#     layers.Dense(128, activation="relu"),
#     layers.Dropout(0.3),
#     layers.Dense(len(class_names), activation="softmax")
# ])

# cnn_model.compile(
#     optimizer="adam",
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"]
# )

# history = cnn_model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=10
# )

# cnn_model.summary()


### CNN evaluation template

For a real classification dataset, a confusion matrix and ROC curve can be generated from the held-out test data. These metrics belong to the CNN classification task, not to Apriori.


In [ ]:
# Optional CNN evaluation template.

# test_ds = tf.keras.utils.image_dataset_from_directory(
#     image_dataset_folder,
#     validation_split=0.20,
#     subset="validation",
#     seed=42,
#     image_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     shuffle=False
# )

# y_true = np.concatenate([y.numpy() for _, y in test_ds])
# y_prob = cnn_model.predict(test_ds)
# y_pred = np.argmax(y_prob, axis=1)

# print(classification_report(y_true, y_pred, target_names=class_names))

# cm = confusion_matrix(y_true, y_pred)

# plt.figure(figsize=(7, 6))
# plt.imshow(cm, cmap="Blues")
# plt.colorbar()
# plt.xticks(range(len(class_names)), class_names, rotation=45)
# plt.yticks(range(len(class_names)), class_names)
# plt.xlabel("Predicted")
# plt.ylabel("Actual")
# plt.title("CNN Confusion Matrix")
# plt.tight_layout()
# plt.show()


## 21. Conclusions

The analysis combines descriptive statistics, feature engineering and Apriori association mining to study pollution and energy-related patterns.

The important points to discuss in the final report are:

1. The dataset contains pollution, waste, energy recovery, renewable energy and economic indicators for multiple countries and years.
2. Missing values and duplicate records were checked before analysis, pollution indices were normalized for preprocessing, and country/year representations were prepared for numerical processing.
3. Numerical variables were converted into Low, Medium and High categories using dataset-based percentile thresholds.
4. Apriori was used to identify combinations of pollution and energy characteristics that occur together.
5. Support, confidence and lift were used to evaluate the strength and usefulness of the association rules.
6. Sensitivity analysis checked how the number of rules changes when the minimum support is changed.
7. A chronological train/test-style split was used as a rule-stability check.
8. Association rules describe co-occurrence patterns; they do **not** establish causation.

### Important note about CNN

A CNN, confusion matrix and ROC curve are not included in this notebook because the supplied dataset contains tabular pollution and energy measurements, not images or delivery outcomes. Adding a CNN to this dataset would not be methodologically appropriate.

If a separate image-based delivery dataset is part of the assignment, the CNN should be implemented as a separate model and compared at the methodology level rather than pretending that Apriori itself has CNN-style accuracy metrics.


## 22. Suggested final report structure

### 1. Introduction
Explain pollution, energy recovery and the purpose of discovering relationships between them.

### 2. Dataset
Describe the countries, years and major pollution/energy variables.

### 3. Data Preprocessing
Discuss duplicate checking, missing-value checking, categorical variables and feature engineering.

### 4. Exploratory Data Analysis
Include distributions, trends and the correlation matrix.

### 5. Apriori Methodology
Explain transactions, frequent itemsets, support, confidence and lift.

### 6. Results
Present the important frequent itemsets and association rules.

### 7. Validation
Discuss support-threshold sensitivity and chronological rule stability.

### 8. Findings
Explain the strongest observed associations without claiming that correlation or association proves causation.

### 9. Limitations
Mention the relatively small number of observations, categorical thresholds and the fact that Apriori identifies associations rather than causal relationships.

### 10. Conclusion
Summarize what the analysis reveals about pollution and energy-related co-occurrence patterns.


---
### End of Analysis

**Project:** Global Pollution Analysis and Energy Recovery  
**Primary algorithm:** Apriori Association Rule Mining  
**Evaluation:** Support, Confidence, Lift, Sensitivity Analysis and Temporal Rule Stability
